# 物理嵌入 TransformerPayne 演示

这个 Notebook 演示如何使用物理嵌入的 TransformerPayne 模型进行恒星光谱生成和物理量分析。

## 目录

1. [环境设置](#环境设置)
2. [模型加载](#模型加载)
3. [光谱生成](#光谱生成)
4. [物理量分析](#物理量分析)
5. [与原始模型对比](#与原始模型对比)

In [ ]:
# 环境设置
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

# 检查设备
print(f"JAX 设备：{jax.devices()}")
print(f"JAX 版本：{jax.__version__}")

In [ ]:
# 导入模型
from transformer_payne_physics import (
    create_physics_embedded_model,
    TransformerPayneModelPhysics,
)
from flax.core.frozen_dict import freeze

## 模型加载

In [ ]:
# 创建物理嵌入模型
model, params = create_physics_embedded_model(
    use_physics=True,
    n_depth_layers=64,
    n_angles=20,
)

print(f"模型已创建")
print(f"参数量：{jax.tree_util.tree_flatten(params)[0][0].size:,}")

## 光谱生成

In [ ]:
# 准备输入
wavelengths = jnp.linspace(4670, 4960, 100)  # Å
log_wavelengths = jnp.log10(wavelengths)

# 太阳参数
solar_params = jnp.array([5777, 4.44, 0.0, 0.0])  # Teff, logg, [Fe/H], [α/Fe]

# 前向传播
variables = {"params": freeze(params)}

output, physics_outputs = model.apply(
    variables,
    (log_wavelengths, solar_params),
    train=False,
    return_physics_outputs=True,
)

print(f"输出形状：{output.shape}")
print(f"强度范围：[{output[:, 0].min():.2e}, {output[:, 0].max():.2e}]")
print(f"连续谱范围：[{output[:, 1].min():.2e}, {output[:, 1].max():.2e}]")

In [ ]:
# 绘制光谱
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(wavelengths, output[:, 0], label='Intensity', linewidth=2)
ax.plot(wavelengths, output[:, 1], label='Continuum', linewidth=2, linestyle='--')

ax.set_xlabel('Wavelength (Å)', fontsize=12)
ax.set_ylabel('Intensity (erg/s/cm³/ster)', fontsize=12)
ax.set_title('Solar Spectrum (Physics-Embedded)', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 物理量分析

In [ ]:
# 查看物理量输出
print("物理量输出:")
print(f"  吸收系数 κ: {physics_outputs['kappa'].shape}")
print(f"  散射系数 σ: {physics_outputs['sigma'].shape}")
print(f"  温度分布 T: {physics_outputs['T'].shape}")
print(f"  光学深度 τ: {physics_outputs['tau'].shape}")

print(f"\n吸收系数范围：[{physics_outputs['kappa'].min():.2e}, {physics_outputs['kappa'].max():.2e}] cm⁻¹")
print(f"散射系数范围：[{physics_outputs['sigma'].min():.2e}, {physics_outputs['sigma'].max():.2e}] cm⁻¹")
print(f"温度范围：[{physics_outputs['T'].min():.1f}, {physics_outputs['T'].max():.1f}] K")

In [ ]:
# 绘制物理量
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 吸收系数
axes[0, 0].plot(wavelengths, physics_outputs['kappa'])
axes[0, 0].set_xlabel('Wavelength (Å)')
axes[0, 0].set_ylabel('κ (cm⁻¹)')
axes[0, 0].set_title('Absorption Coefficient')
axes[0, 0].set_yscale('log')
axes[0, 0].grid(True, alpha=0.3)

# 散射系数
axes[0, 1].plot(wavelengths, physics_outputs['sigma'])
axes[0, 1].set_xlabel('Wavelength (Å)')
axes[0, 1].set_ylabel('σ (cm⁻¹)')
axes[0, 1].set_title('Scattering Coefficient')
axes[0, 1].set_yscale('log')
axes[0, 1].grid(True, alpha=0.3)

# 温度分布
depth_layers = np.arange(physics_outputs['T'].shape[0])
axes[1, 0].plot(physics_outputs['T'], depth_layers)
axes[1, 0].set_xlabel('Temperature (K)')
axes[1, 0].set_ylabel('Depth Layer')
axes[1, 0].set_title('Temperature Profile')
axes[1, 0].invert_yaxis()
axes[1, 0].grid(True, alpha=0.3)

# 光学深度
tau_mean = jnp.mean(physics_outputs['tau'], axis=1)
axes[1, 1].plot(tau_mean, depth_layers)
axes[1, 1].set_xlabel('Optical Depth (mean)')
axes[1, 1].set_ylabel('Depth Layer')
axes[1, 1].set_title('Optical Depth Profile')
axes[1, 1].set_xscale('log')
axes[1, 1].invert_yaxis()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 不同恒星参数的光谱

In [ ]:
# 批量生成不同参数的光谱
from jax import vmap

# 定义一组恒星参数
stellar_params_batch = jnp.array([
    [5777, 4.44, 0.0, 0.0],    # 太阳
    [6000, 4.5, -0.5, 0.2],   # 较热，贫金属
    [4500, 4.0, -1.0, 0.4],   # 较冷，极贫金属
    [7000, 4.2, 0.3, 0.0],    # 很热，富金属
])

# 批量应用
def apply_single(params):
    spec, _ = model.apply(
        variables,
        (log_wavelengths, params),
        train=False,
        return_physics_outputs=True,
    )
    return spec

batch_apply = vmap(apply_single, in_axes=0, out_axes=0)
spectra_batch = batch_apply(stellar_params_batch)

print(f"批量输出形状：{spectra_batch.shape}")

In [ ]:
# 绘制对比图
fig, ax = plt.subplots(figsize=(14, 7))

labels = [
    'Sun (5777K)',
    'Hot, Metal-poor (6000K)',
    'Cool, Very metal-poor (4500K)',
    'Very hot, Metal-rich (7000K)',
]

for i, (params, spec, label) in enumerate(zip(stellar_params_batch, spectra_batch, labels)):
    ax.plot(wavelengths, spec[:, 0], label=label, linewidth=2, alpha=0.8)

ax.set_xlabel('Wavelength (Å)', fontsize=12)
ax.set_ylabel('Intensity', fontsize=12)
ax.set_title('Stellar Spectra with Different Parameters', fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 梯度检查

In [ ]:
# 检查梯度 (验证可微分性)
from jax import grad

def intensity_fn(params):
    spec, _ = model.apply(
        variables,
        (log_wavelengths, params),
        train=False,
        return_physics_outputs=True,
    )
    return jnp.mean(spec[:, 0])  # 平均强度

grad_fn = grad(intensity_fn)
gradients = grad_fn(solar_params)

print("梯度 (d<I>/dparams):")
print(f"  d<I>/dTeff = {gradients[0]:.6e}")
print(f"  d<I>/dlogg = {gradients[1]:.6e}")
print(f"  d<I>/d[Fe/H] = {gradients[2]:.6e}")
print(f"  d<I>/d[α/Fe] = {gradients[3]:.6e}")

print(f"\n梯度是否有限：{jnp.all(jnp.isfinite(gradients))}")

## 总结

本演示展示了物理嵌入 TransformerPayne 的核心功能：

✅ 光谱生成  
✅ 物理量输出 (κ, σ, T, τ)  
✅ 批量推理  
✅ 梯度计算  

下一步：
- 使用真实 PHOENIX 数据集训练
- 与原始模型进行定量对比
- 参数反演实验